# 第22讲：独立验证发布、同步与恢复
本讲提供旧版与增加连接池检查的教学 Skill，使用原生共享和验证接口。也可以选择第21讲的实际产物。
CLI 与 Notebook 共用同一个 run.sh。每次运行使用独立目录，默认关闭模型调用。
任务先用旧版执行，再在同步后的新会话中处理同题，最后恢复旧内容。


In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'examples/22-skillclaw-shared-revision/run.sh').is_file())
sys.path.insert(0, str(ROOT / 'examples'))
from skillclaw_lab.runtime import run_cli
CLI = ROOT / 'examples/22-skillclaw-shared-revision/run.sh'


In [ ]:
# 离线测试与独立输入准备都不连接模型。
_ = run_cli(CLI, ['--offline'], timeout=90)
_ = run_cli(CLI, ['--standalone', '--prepare-only'], timeout=30)


In [ ]:
ALLOW_PAID = False
SOURCE_RUN = None  # 默认使用本讲独立输入。可填第21讲已完成的真实 run。
PUBLISH_MODE = 'direct'  # 独立输入可另选 validated。
SYNC_MODE = 'poll'  # 手动拉取可选 manual。
if ALLOW_PAID:
    if SOURCE_RUN is None:
        arguments = ['--standalone', '--publish-mode', PUBLISH_MODE]
    else:
        arguments = ['--source-run', str(Path(SOURCE_RUN).expanduser().resolve())]
    _ = run_cli(CLI, [*arguments, '--sync-mode', SYNC_MODE])
else:
    print('模型调用尚未启用。先核对准备目录中的 provided.json 与两份 Skill。')


## 从终端给出的 REPORT.md 查看结果
先从 paired_tasks 对照 baseline-pool/pool 和 baseline-upstream/upstream 的快照、Session 和工具顺序。
分别看正式文件、接收端文件、skill_view 实际读取、完整查询回执和条件动作。上游异常仍查询连接池就是偏差，不能因 PRM 通过而改判。
每项任务的 input.json 保存实际 system/user 提示。两种现场使用相同的一轮初查要求，按 Skill 正文选择分支，不把全部工具当成必做清单；工具权限没有按场景删减。
恢复是重新 push 来源保存的旧内容、同步并启动独立新会话，不是 restore hermes，也不换成仓库当前 seed。恢复后的查询只记录观察，不宣称行为退回或能力收益。
同一来源再次执行会创建另一个独立副本并再次付费，不是无费用恢复运行。请先读已有报告，不要重复执行在线单元格。